# Coupler Bayesian optimization

One candidate coupler Z is evaluated with a full SWAP time trace. Optimize P01 contrast with P10/P01 anticorrelation; validate gate fidelity separately.

Edit the JSON experiment specification and check source calibrations before running hardware. The notebook does not publish a calibration.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'execute':
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise FileNotFoundError('Start Jupyter in Platform_LQ or execute')
sys.path.insert(0, str(ROOT))
from agent_cli import read_spec, preflight, run
SPEC = ROOT / 'config/plans/coupler_bo_example.json'
RUN_HARDWARE = True
spec = read_spec(SPEC)
preflight(spec)


In [ ]:
if RUN_HARDWARE:
    from src.progress import watch_run
    with watch_run(ROOT / "result" / "coupler", preflight(spec)['jobs'],
                   run_mode=spec.get("run_mode", "new"), resume_run_tag=spec.get("resume_run_tag", ""),
                   title='耦合器贝叶斯优化'):
        outcome = run(spec)
        print(outcome)
else:
    print('Dry plan only; no LQCloud submission.')


## Inspect saved BO results

Set PLOT_RUN_TAG to one saved run directory to draw figures offline. A hardware run in the cell above is plotted automatically. Z-frequency generates a separate GP and two figures per Z bias; coupler plots its SWAP contrast and anticorrelation score. Figures are stored under each run's fig/ directory.


In [ ]:
from optimize.show_fig import plot_bo_run

PLOT_RUN_TAG = ""  # For offline replay, paste one existing run directory name here.
run_dir = True
if RUN_HARDWARE:
    run_dir = Path(outcome["run_dir"])
elif PLOT_RUN_TAG:
    if (Path(PLOT_RUN_TAG).name != PLOT_RUN_TAG
            or "/" in PLOT_RUN_TAG or '\\' in PLOT_RUN_TAG):
        raise ValueError("PLOT_RUN_TAG must be one result directory name")
    run_dir = ROOT / "result" / 'coupler' / PLOT_RUN_TAG

if run_dir is None:
    print("No data: set PLOT_RUN_TAG to a saved run or enable RUN_HARDWARE.")
else:
    figures = plot_bo_run(run_dir)
